## 🧾 Analyse der Rohdaten

Die ursprünglichen Daten liegen in mehreren **relationalen Tabellen** vor (z. B. `results`, `races`, `drivers`, `constructors`, `circuits`).  
Diese Tabellen wurden zunächst **einzeln geladen** und hinsichtlich **Größe**, **Struktur** und **Schlüsselattribute** untersucht.

### 🔍 Überprüft wurde:
- Welche Tabellen **welche Informationen** enthalten  
- Wie die Tabellen über **IDs miteinander verknüpft** sind  
- Ob die **Granularität der Daten** zur Problemstellung passt  

---

## 🔗 Zusammenführung der Tabellen

Als **Basis** wurde die Tabelle `results` gewählt, da sie bereits **eine Zeile pro Fahrer und Rennen** enthält.  
Weitere Tabellen wurden über **Schlüsselattribute** (`raceId`, `driverId`, `constructorId`, `circuitId`) mittels **Joins** ergänzt.

### 📦 Das Ergebnis ist ein konsistenter Datensatz mit:
- Fahrer-, Team- und Streckeninformationen  
- Rennkontext (*Saison*, *Runde*, *Datum*)  
- Startposition und **Rennergebnis**


In [81]:
import pandas as pd
import numpy as np


In [82]:
races = pd.read_csv("../data/raw/races.csv")
results = pd.read_csv("../data/raw/results.csv")
drivers = pd.read_csv("../data/raw/drivers.csv")
constructors = pd.read_csv("../data/raw/constructors.csv")
circuits = pd.read_csv("../data/raw/circuits.csv")

print("races", races.shape)
print("results", results.shape)


races (1125, 18)
results (26759, 18)


In [83]:
df = results.copy()
print("df start:", df.shape)
df.head()


df start: (26759, 18)


,resultId,raceId,driverId,constructorId,number,grid,position,positionText,positionOrder,points,laps,time,milliseconds,fastestLap,rank,fastestLapTime,fastestLapSpeed,statusId
0,1,18,1,1,22,1,1,1,1,10.0,58,1:34:50.616,5690616,39,2,1:27.452,218.300,1
1,2,18,2,2,3,5,2,2,2,8.0,58,+5.478,5696094,41,3,1:27.739,217.586,1
2,3,18,3,3,7,7,3,3,3,6.0,58,+8.163,5698779,41,5,1:28.090,216.719,1
3,4,18,4,4,5,11,4,4,4,5.0,58,+17.181,5707797,58,7,1:28.603,215.464,1
4,5,18,5,1,23,3,5,5,5,4.0,58,+18.014,5708630,43,1,1:27.418,218.385,1


In [84]:
races_small = races[["raceId", "year", "round", "circuitId", "date"]].copy()
races_small = races_small.rename(columns={"year": "season", "date": "race_date"})
df = df.merge(races_small, on="raceId", how="left")
print("after races merge:", df.shape)


after races merge: (26759, 22)


In [85]:
circuits_small = circuits[["circuitId", "name"]].rename(columns={"name": "circuit_name"})
df = df.merge(circuits_small, on="circuitId", how="left")
print("after circuits merge:", df.shape)


after circuits merge: (26759, 23)


In [86]:
drivers_small = drivers[["driverId", "forename", "surname"]].copy()
drivers_small["driver_name"] = drivers_small["forename"] + " " + drivers_small["surname"]
drivers_small = drivers_small[["driverId", "driver_name"]]
df = df.merge(drivers_small, on="driverId", how="left")
print("after drivers merge:", df.shape)


after drivers merge: (26759, 24)


In [87]:
constructors_small = constructors[["constructorId", "name"]].rename(columns={"name": "constructor_name"})
df = df.merge(constructors_small, on="constructorId", how="left")
print("after constructors merge:", df.shape)


after constructors merge: (26759, 25)


## 🧹 Bereinigung & Plausibilitätsprüfung

Im Rahmen der **strukturellen EDA** wurden folgende Schritte durchgeführt:

- 🔢 Numerische Spalten (z. B. `grid`, `positionOrder`) wurden **explizit konvertiert**  
- ❌ Einträge mit **fehlenden oder ungültigen Werten** wurden entfernt  
- 🚫 **Unrealistische Startpositionen** (z. B. `grid = 0`) wurden ausgeschlossen  

---

### 🎯 Definition der Zielvariable

Die Zielvariable wurde auf Basis der **Zielposition** wie folgt definiert:

- `1` → Fahrer **beendet das Rennen in den Top 10**  
- `0` → Fahrer **außerhalb der Top 10**


In [88]:
df["grid"] = pd.to_numeric(df["grid"], errors="coerce")
df["positionOrder"] = pd.to_numeric(df["positionOrder"], errors="coerce")

df = df.dropna(subset=["season", "round", "raceId", "race_date",
                       "circuitId", "driverId", "constructorId",
                       "grid", "positionOrder"])

df = df[df["grid"] > 0]
print("after cleaning:", df.shape)


after cleaning: (25121, 25)


In [89]:
df["target_top10"] = (df["positionOrder"] <= 10).astype(int)

print("target distribution:")
print(df["target_top10"].value_counts())
print(df["target_top10"].value_counts(normalize=True))


target distribution:
target_top10
0    13814
1    11307
Name: count, dtype: int64
target_top10
0    0.549898
1    0.450102
Name: proportion, dtype: float64


In [90]:
base_cols = [
    "season", "round", "raceId", "race_date",
    "circuitId", "circuit_name",
    "driverId", "driver_name",
    "constructorId", "constructor_name",
    "grid", "positionOrder", "statusId",
    "target_top10"
]
base_cols = [c for c in base_cols if c in df.columns]

df_base = df[base_cols].copy()
df_base = df_base.sort_values(["season", "round", "raceId", "driverId"]).reset_index(drop=True)

df_base.to_csv("../data/processed/driver_race_base.csv", index=False)
print("Saved base:", df_base.shape)

df_base.head()


Saved base: (25121, 14)


,season,round,raceId,race_date,circuitId,circuit_name,driverId,driver_name,constructorId,constructor_name,grid,positionOrder,statusId,target_top10
0,1950,1,833,1950-05-13,9,Silverstone Circuit,579,Juan Fangio,51,Alfa Romeo,3,12,44,0
1,1950,1,833,1950-05-13,9,Silverstone Circuit,589,Louis Chiron,105,Maserati,11,18,8,0
2,1950,1,833,1950-05-13,9,Silverstone Circuit,619,Bob Gerard,151,ERA,13,6,13,1
3,1950,1,833,1950-05-13,9,Silverstone Circuit,627,Louis Rosier,154,Talbot-Lago,9,5,12,1
4,1950,1,833,1950-05-13,9,Silverstone Circuit,640,Toulo de Graffenried,105,Maserati,8,17,5,0
